In [180]:
classes_nsfw = ["drawings", "hentai", "neutral", "porn", "sexy"]
classes_lv1 = ["zhengchang", "shezheng", "baokong", "weijin", "seqing", "xinggan"]
classes_3 = ["normal", "politics", "porn"]

In [181]:
# collect shumei
import os
import json
from os.path import expanduser
import pandas as pd

df = pd.DataFrame()

model = "groundtruth"

with open(
    expanduser("/mnt/storage/user/wangruohui/涉政与色情低俗图包/ann_dana.txt")
) as f:
    for line in f:
        name, label = line.strip().rsplit(" ", 1)
        label = classes_3[int(label)]

        df.loc[name, model] = label


model = "shumei"

with open(expanduser("~/涉政与色情低俗图包/shumei-label.jsonl")) as f:
    # policount = 0
    # policorrect = 0
    # porncount = 0
    # porncorrect = 0
    corrects = {
        "poli": {"text": 0, "image": 0,"normal": 0},
        "porn": {"text": 0, "image": 0,"normal": 0},
    }  # zheng/huang, text/image
    count = {
        "poli": {"text": 0, "image": 0,"normal": 0},
        "porn": {"text": 0, "image": 0,"normal": 0},
    }  # zheng/huang, text/image
    for line in f:
        d = json.loads(line)
        name = d["img"]
        pred_name = d["riskLabel1"]
        source = d["riskSource"]
        correct = pred_name == df.loc[name, "groundtruth"]

        df.loc[name, model] = pred_name
        df.loc[name, f"{model}-check"] = correct
        df.loc[name, f"{model}-source"] = source

        if "涉政" in name:
            type_ = "poli"
        elif "色情" in name:
            type_ = "porn"
        else:
            raise ValueError('name', name)

        if source == 1001:
            source = "text"
        elif source == 1002:
            source = "image"
        elif source == 1000:
            source = "normal"
        else:
            raise ValueError('source', source)

        if correct:
            corrects[type_][source] += 1
        count[type_][source] += 1

    df.loc["涉政图片正确率", f"{model}-check"] = corrects['poli']["image"] / count['poli']["image"]
    df.loc["涉政文本正确率", f"{model}-check"] = corrects['poli']["text"] / count['poli']["text"]
    df.loc["涉政正常正确率", f"{model}-check"] = corrects['poli']["normal"] / count['poli']["normal"]
    df.loc["涉政正确率", f"{model}-check"] = sum(corrects['poli'].values()) / sum(count['poli'].values())
    df.loc["色情图片正确率", f"{model}-check"] = corrects['porn']["image"] / count['porn']["image"]
    df.loc["色情文本正确率", f"{model}-check"] = corrects['porn']["text"] / count['porn']["text"]
    df.loc["色情正常正确率", f"{model}-check"] = corrects['porn']["normal"] / count['porn']["normal"]
    df.loc["色情正确率", f"{model}-check"] = sum(corrects['porn'].values()) / sum(count['porn'].values())
    df.loc["平均正确率", f"{model}-check"] = sum(sum(v.values()) for v in corrects.values()) / sum(sum(v.values()) for v in count.values())

df = pd.concat([ df.iloc[-9:,:], df.iloc[:-9,:]], axis=0)

In [182]:
import pickle
from pathlib import Path
import pandas as pd


test_paths = [
    "1009-convnext-v2-base_32xb32_huangfan2-384px/test-dana-25.pkl",
    "1009-convnext-v2-base_32xb32_huangfan2-384px/test-dana-40.pkl",
    "1009-convnext-v2-base_32xb32_huangfan2-384px/test-dana-45.pkl",
    "1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/test-dana-55.pkl",
    "1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/test-dana-60.pkl",
    "1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/test-dana-65.pkl",
]

for f in test_paths:
    # policount = 0
    # policorrect = 0
    # porncount = 0
    # porncorrect = 0
    corrects = {
        "poli": {"text": 0, "image": 0, "normal": 0},
        "porn": {"text": 0, "image": 0, "normal": 0},
    }  # zheng/huang, text/image
    count = {
        "poli": {"text": 0, "image": 0, "normal": 0},
        "porn": {"text": 0, "image": 0, "normal": 0},
    }  # zheng/huang, text/image

    model = f.split(".")[0].replace("test-dana-", "")

    f = Path("work_dirs") / f
    if "smooth" in f.name:
        continue

    data = pickle.load(f.open("rb"))

    d = data[0]
    num_classes = d["num_classes"]
    if num_classes == 5 or num_classes == 1000:
        classes = classes_nsfw
        model = "涉黄" + model
    elif num_classes == 6:
        classes = classes_lv1
        model = "涉政涉黄" + model
    elif num_classes == 3:
        classes = classes_3
    else:
        raise Exception(f"Unknown num_classes: {repr(num_classes)}")

    for d in data:
        name = d["img_path"].lstrip("/mnt/storage/user/wangruohui//")
        name = name[len("涉政与色情低俗图包/") :]
        gt_label = int(d["gt_label"])

        pred_label = int(d["pred_label"])
        pred_name = classes[pred_label]
        correct = (pred_name==df.loc[name, "groundtruth"])
        source = df.loc[name, "shumei-source"]

        df.loc[name, model] = pred_name
        df.loc[name, f"{model}-check"] = correct

        if "涉政" in name:
            type_ = "poli"
        elif "色情" in name:
            type_ = "porn"
        else:
            raise ValueError('name', name)

        if source == 1001:
            source = "text"
        elif source == 1002:
            source = "image"
        elif source == 1000:
            source = "normal"
        else:
            raise ValueError('source', source)

        if correct:
            corrects[type_][source] += 1
        count[type_][source] += 1

    df.loc["涉政图片正确率", f"{model}-check"] = corrects['poli']["image"] / count['poli']["image"]
    df.loc["涉政文本正确率", f"{model}-check"] = corrects['poli']["text"] / count['poli']["text"]
    df.loc["涉政正常正确率", f"{model}-check"] = corrects['poli']["normal"] / count['poli']["normal"]
    df.loc["涉政正确率", f"{model}-check"] = sum(corrects['poli'].values()) / sum(count['poli'].values())
    df.loc["色情图片正确率", f"{model}-check"] = corrects['porn']["image"] / count['porn']["image"]
    df.loc["色情文本正确率", f"{model}-check"] = corrects['porn']["text"] / count['porn']["text"]
    df.loc["色情正常正确率", f"{model}-check"] = corrects['porn']["normal"] / count['porn']["normal"]
    df.loc["色情正确率", f"{model}-check"] = sum(corrects['porn'].values()) / sum(count['porn'].values())
    df.loc["平均正确率", f"{model}-check"] = sum(sum(v.values()) for v in corrects.values()) / sum(sum(v.values()) for v in count.values())

    df = df.copy()

In [183]:
df

,groundtruth,shumei,shumei-check,shumei-source,1009-convnext-v2-base_32xb32_huangfan2-384px/25,1009-convnext-v2-base_32xb32_huangfan2-384px/25-check,1009-convnext-v2-base_32xb32_huangfan2-384px/40,1009-convnext-v2-base_32xb32_huangfan2-384px/40-check,1009-convnext-v2-base_32xb32_huangfan2-384px/45,1009-convnext-v2-base_32xb32_huangfan2-384px/45-check,1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/55,1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/55-check,1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/60,1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/60-check,1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/65,1009-convnext-v2-tiny_8xb48_huangfan2-noscale-384px/65-check
涉政图片正确率,NaN,NaN,1.0,NaN,NaN,0.75,NaN,0.714286,NaN,0.714286,NaN,0.696429,NaN,0.714286,NaN,0.714286
涉政文本正确率,NaN,NaN,1.0,NaN,NaN,0.560976,NaN,0.609756,NaN,0.609756,NaN,0.682927,NaN,0.682927,NaN,0.682927
涉政正常正确率,NaN,NaN,0.2,NaN,NaN,0.7,NaN,0.6,NaN,0.6,NaN,0.6,NaN,0.6,NaN,0.6
涉政正确率,NaN,NaN,0.925234,NaN,NaN,0.672897,NaN,0.663551,NaN,0.663551,NaN,0.682243,NaN,0.691589,NaN,0.691589
色情图片正确率,NaN,NaN,0.950617,NaN,NaN,0.962963,NaN,0.962963,NaN,0.962963,NaN,0.975309,NaN,0.975309,NaN,0.962963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
色情低俗图包/uZmo2k.jpg,porn,porn,True,1002.0,porn,True,porn,True,porn,True,porn,True,porn,True,porn,True
色情低俗图包/uZp8d3.jpg,porn,porn,True,1002.0,porn,True,porn,True,porn,True,porn,True,porn,True,porn,True
色情低俗图包/uZpHGz.jpg,porn,porn,True,1002.0,porn,True,porn,True,porn,True,porn,True,porn,True,porn,True
色情低俗图包/图片10.png,porn,porn,True,1002.0,porn,True,porn,True,porn,True,porn,True,porn,True,porn,True


In [184]:
import random
import argparse
import io
import xlsxwriter
import json
from PIL import Image, ImageDraw

WRITE_IMAGE = True

workbook = xlsxwriter.Workbook("test-1009.xlsx")

worksheet = workbook.add_worksheet("test-1009")
worksheet.freeze_panes(1, 0)

if WRITE_IMAGE:
    worksheet.set_column(0, 1, 40)

worksheet.write(0, 0, "image_path")
worksheet.write(0, 1, "image")

# nama_map = {
#     "drawings": "正常",
#     "neutral": "正常",
#     "zhengchang": "正常",
#     "shezheng": "涉政",
#     "politics": "涉政",
#     "baokong": "暴恐",
#     "seqing": "涉黄",
#     "porn": "涉黄",
#     "hentai":"涉黄",
#     "xinggan": "性感",
#     "sexy": "性感",
#     "weijin": "违禁",
# }

for i, col_name in enumerate(df.columns):
    worksheet.write(
        0, i + 1 + WRITE_IMAGE, col_name, workbook.add_format({"text_wrap": True})
    )

for i, row in enumerate(df.iterrows()):
    worksheet.write(i + 1, 0, row[0])
    for j, item in enumerate(row[1]):
        # item = nama_map[item]
        if not pd.isna(item):
            worksheet.write(i + 1, j + 1 + WRITE_IMAGE, item)

    if WRITE_IMAGE and ("/" in row[0]):
        img = Image.open(f"/mnt/storage/user/wangruohui/涉政与色情低俗图包/{row[0]}")
        img.thumbnail((150, 120))
        cache_fn = f"/tmp/{row[0]}"
        img.save(cache_fn)
        worksheet.embed_image(i + 1, 1, cache_fn)
        worksheet.set_row_pixels(i + 1, 100)

        print(cache_fn)

workbook.close()

/tmp/涉政图包/100.jfif
/tmp/涉政图包/10.jfif
/tmp/涉政图包/11.jfif
/tmp/涉政图包/12.jfif
/tmp/涉政图包/13.jfif
/tmp/涉政图包/14.jfif
/tmp/涉政图包/15.jfif
/tmp/涉政图包/16.jpg
/tmp/涉政图包/17.jfif
/tmp/涉政图包/18.jfif
/tmp/涉政图包/19.jfif
/tmp/涉政图包/1.jpg
/tmp/涉政图包/20.jfif
/tmp/涉政图包/21.jfif
/tmp/涉政图包/22.jfif
/tmp/涉政图包/23.jfif
/tmp/涉政图包/24.jfif
/tmp/涉政图包/25.jfif
/tmp/涉政图包/26.jfif
/tmp/涉政图包/27.jfif
/tmp/涉政图包/28.jfif
/tmp/涉政图包/29.jfif
/tmp/涉政图包/2.jfif
/tmp/涉政图包/30.jfif
/tmp/涉政图包/31.jfif
/tmp/涉政图包/32.jfif
/tmp/涉政图包/33.jfif
/tmp/涉政图包/34.jfif
/tmp/涉政图包/35.jfif
/tmp/涉政图包/36.jfif
/tmp/涉政图包/37.jfif
/tmp/涉政图包/38.jfif
/tmp/涉政图包/39.jfif
/tmp/涉政图包/3.jpg
/tmp/涉政图包/40.jfif
/tmp/涉政图包/41.jfif
/tmp/涉政图包/42.jfif
/tmp/涉政图包/43.jfif
/tmp/涉政图包/44.jfif
/tmp/涉政图包/45.jfif
/tmp/涉政图包/46.jfif
/tmp/涉政图包/47.jfif
/tmp/涉政图包/48.jfif
/tmp/涉政图包/49.jfif
/tmp/涉政图包/4.jfif
/tmp/涉政图包/50.jfif
/tmp/涉政图包/51.jfif
/tmp/涉政图包/52.jfif
/tmp/涉政图包/53.png
/tmp/涉政图包/54.png
/tmp/涉政图包/55l.png
/tmp/涉政图包/56.jfif
/tmp/涉政图包/56.png
/tmp/涉政图包/57.jfif
/tmp/涉政图包/58.jfif
/tmp/涉政图包/59.jfif
/

In [185]:
cache_fn

'/tmp/色情低俗图包/图片9.png'